# Merge: named tracking cache + team assignment -> one match cache
Run after the team assigner notebook (same session, or add its output to a dataset and set `TEAM_PATH`).
Keeps both inputs untouched; the merged file is rebuilt in about a minute whenever either input changes.

In [ ]:
# ============================================================================================
# Merge the named tracking cache + the team assignment cache into ONE match cache
#  * one box per (frame, person): if a person has 2 boxes in a frame, the real / most confident one is kept as the
#    box and the other is stored inside it as `duplicate_boxes` -> unique (frame, id), and still no box is lost
#  * off-pitch / extra people (class "other") go to a separate per-frame list `others`, so analytics loop over the
#    25 people only
#  * team (0/1 from the kit clustering) is written on every box; the full team structure is kept under "team"
#  * checks: every id has a team, names vs kits agree (if names exist), unique (frame, id), boxes in == boxes out
# ============================================================================================
import os
import pickle
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

TRACKING_PATH = "/kaggle/input/datasets/zeinsaad/deep-eiou-full-match/barca_atletico_tracking_cache_named (4).pkl"
TEAM_PATH     = "/kaggle/working/barca_atletico_team_assignment_stable.pkl"     # normal structure (team notebook)
OUT_PATH      = "/kaggle/working/barca_atletico_match_cache.pkl"
OUT_TABLE     = "/kaggle/working/barca_atletico_match_tracks.parquet"
PEOPLE = ("player", "goalkeeper", "referee")

with open(TRACKING_PATH, "rb") as fh:
    tr = pickle.load(fh)
with open(TEAM_PATH, "rb") as fh:
    tm = pickle.load(fh)
cache, locked, pinfo = tr["tracking_cache"], tr["locked_class_by_id"], tr.get("player_info", {})
team_by_id = {k: v for k, v in tm["team_by_id"].items() if v is not None}
print(f"Tracking: {len(cache)} frames | team cache keys: {sorted(tm)}")

# ---------------------------------------------------------------- checks: ids and teams
people = {tid for tid, c in locked.items() if c in PEOPLE}
need_team = sorted(t for t in people if locked[t] != "referee")
missing = [t for t in need_team if t not in team_by_id]
unknown = sorted(set(team_by_id) - people)
print(f"People: {len(people)} ({Counter(locked[t] for t in people)}) | with a team: "
      f"{len(need_team) - len(missing)}/{len(need_team)}")
if missing:
    print(f"  [warn] no team for ids {missing} -> team None in the merged cache")
if unknown:
    print(f"  [warn] team cache has ids not in the tracking cache: {unknown} -> team and tracking caches are from "
          f"different runs?")
print(f"Team sizes: {Counter(team_by_id[t] for t in need_team if t in team_by_id)}")

team_names = dict(tm.get("team_names", {}))
_pairs = Counter((team_by_id[t], pinfo[t]["team"]) for t in need_team
                 if t in team_by_id and pinfo.get(t, {}).get("team"))
if _pairs:                                                   # names exist -> cross-check kits vs names
    for k in (0, 1):
        cands = {code: n for (kk, code), n in _pairs.items() if kk == k}
        if cands and k not in team_names:
            team_names[k] = max(cands, key=cands.get)
    bad = [t for t in need_team if t in team_by_id and pinfo.get(t, {}).get("team")
           and team_names.get(team_by_id[t]) != pinfo[t]["team"]]
    print(f"Kits vs names: {len(need_team) - len(bad)}/{len(need_team)} agree | team names {team_names}")
    for t in bad:
        print(f"  [DISAGREE] id {t} {pinfo[t].get('name')}: kit team {team_by_id[t]} "
              f"({team_names.get(team_by_id[t])}) vs named {pinfo[t]['team']}")
for k in (0, 1):
    team_names.setdefault(k, f"team {k}")


def source(t):
    return ("interpolated" if t.get("interpolated") else "auto" if t.get("auto_assigned")
            else "recovered" if t.get("recovered") else "tracked")


# ---------------------------------------------------------------- merge
merged = {}
n_in = n_out = n_dup = n_other = 0
for f in sorted(cache):
    d = cache[f]
    by_id, others = defaultdict(list), []
    for t in d["tracks"]:
        n_in += 1
        tid = t["track_id"]
        cls = locked.get(tid, t.get("class"))
        if cls not in PEOPLE:
            others.append({"id": tid, "bbox": [float(v) for v in t["bbox"]], "conf": float(t["conf"]),
                           "kind": pinfo.get(tid, {}).get("name", "other"), "raw_track_id": t.get("raw_track_id")})
            n_other += 1
            continue
        by_id[tid].append(t)
    tracks = []
    for tid, lst in by_id.items():
        lst.sort(key=lambda t: (bool(t.get("duplicate")), bool(t.get("interpolated")), -float(t["conf"])))
        p = lst[0]
        info = pinfo.get(tid, {})
        team = team_by_id.get(tid)
        rec = {"track_id": tid, "bbox": [float(v) for v in p["bbox"]], "conf": float(p["conf"]),
               "class": locked.get(tid, p.get("class")), "team": team,
               "team_name": team_names.get(team) if team is not None else None,
               "name": info.get("name", p.get("name")), "number": info.get("number", p.get("number")),
               "source": source(p), "raw_track_id": p.get("raw_track_id")}
        if p.get("assign_score") is not None:
            rec["assign_score"] = p["assign_score"]
        if len(lst) > 1:
            rec["duplicate_boxes"] = [{"bbox": [float(v) for v in t["bbox"]], "conf": float(t["conf"]),
                                       "raw_track_id": t.get("raw_track_id")} for t in lst[1:]]
            n_dup += len(lst) - 1
        tracks.append(rec)
        n_out += 1
    merged[f] = {"ball": d.get("ball"), "tracks": tracks, "others": others}

# ---------------------------------------------------------------- verify
assert n_in == n_out + n_dup + n_other, "box count mismatch"
assert all(len({t["track_id"] for t in d["tracks"]}) == len(d["tracks"]) for d in merged.values())
print(f"\nBoxes in: {n_in:,} = people boxes {n_out:,} + duplicates kept inside them {n_dup:,} + others {n_other:,}")
print("Unique (frame, id): OK | no box lost: OK")

team_block = {k: v for k, v in tm.items()}
team_block["team_names"] = team_names
out = {"format": "match_cache_v1",
       "tracking_cache": merged,
       "locked_class_by_id": {t: c for t, c in locked.items() if c in PEOPLE},
       "player_info": {t: v for t, v in pinfo.items() if t in people},
       "team": team_block}
with open(OUT_PATH, "wb") as fh:
    pickle.dump(out, fh)

rows = [(f, t["track_id"], t["class"], t["team"], t["team_name"], t["name"], t["number"], *t["bbox"], t["conf"],
         t["source"], len(t.get("duplicate_boxes", ())))
        for f, d in merged.items() for t in d["tracks"]]
df = pd.DataFrame(rows, columns=["frame", "track_id", "class", "team", "team_name", "name", "number",
                                 "x1", "y1", "x2", "y2", "conf", "source", "n_duplicates"])
df["team"] = df["team"].astype("Int64"); df["number"] = pd.to_numeric(df["number"], errors="coerce").astype("Int64")
df.to_parquet(OUT_TABLE, index=False)
print(f"\nSaved {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e6:.0f} MB) and {OUT_TABLE} ({len(df):,} rows)")
print(df.groupby(["class", "team_name"], dropna=False).track_id.nunique().to_string())
